# Structured streaming Demo

In [ ]:
# Imports cell

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, BooleanType

from pyspark.sql.functions import window, col, count

# Convert timestamp (Unix epoch in seconds) to timestamp type
from pyspark.sql.functions import from_unixtime




In [ ]:
# initialize the spark session
spark = SparkSession.builder \
    .appName("KafkaStreamingPrototype") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.0") \
    .getOrCreate()


In [ ]:
# Define the initial raw streaming dataframe
raw_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:19092") \
    .option("subscribe", "wikistreams") \
    .option("startingOffsets", "earliest") \
    .load()


In [ ]:
# Define a simplified schema - just the key fields for analysis
simple_wikimedia_schema = StructType([
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("comment", StringType(), True),
    StructField("length", StructType([
        StructField("old", IntegerType(), True),
        StructField("new", IntegerType(), True)
    ]), True)
])

# Apply the schema to organize the raw input
parsed_df = raw_df.select(
    from_json(col("value").cast("string"), simple_wikimedia_schema).alias("data")
).select("data.*")

In [ ]:
# A simple aggregation of raw data by number of bot edits
windowed_counts_df = parsed_df \
    .withColumn("event_time", from_unixtime(col("timestamp"))) \
    .groupBy(
        window(col("event_time"), "1 hour"),
        col("bot")
    ) \
    .agg(count("*").alias("edit_count")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count")
    )

In [ ]:

# Use console output (goes to Docker logs)
streaming_query_df = windowed_counts_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

# Lab explorations

The cells below extend the demo (TO DO 10 and 12):

1. **Tumbling window** (1 minute): size of the edits, bots vs humans
2. **Overlapping (sliding) window** (5 minutes, sliding every 1 minute): most active pages
3. **Different window time** (10 minutes, tumbling): activity by type of change

Every result has an `exploration` column so the outputs can be told apart in the Docker logs.

In [ ]:
from pyspark.sql import functions as F

# Shared base for all the explorations
# - event_time: real timestamp built from the Unix epoch (required for watermarking)
# - edit_size: bytes added (positive) or removed (negative) by the change
# - watermark: events arriving more than 2 minutes late are dropped,
#   which lets Spark close old windows instead of keeping them in memory forever
events_df = parsed_df \
    .withColumn("event_time", F.timestamp_seconds(F.col("timestamp"))) \
    .withColumn("edit_size", F.col("length.new") - F.col("length.old")) \
    .withWatermark("event_time", "2 minutes")

## Exploration 1 - Tumbling window: size of the edits

Windows of 1 minute that do **not** overlap: each event belongs to exactly one window.
For each window, we compare the edits made by bots and by humans.

In [ ]:
edit_size_tumbling_df = events_df \
    .filter(F.col("type") == "edit") \
    .groupBy(
        F.window(F.col("event_time"), "1 minute"),
        F.col("bot")
    ) \
    .agg(
        F.count("*").alias("edit_count"),
        F.round(F.avg("edit_size"), 1).alias("avg_edit_size"),
        F.max("edit_size").alias("biggest_addition"),
        F.min("edit_size").alias("biggest_removal")
    ) \
    .select(
        F.lit("1_tumbling_edit_size").alias("exploration"),
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        F.col("bot"),
        F.col("edit_count"),
        F.col("avg_edit_size"),
        F.col("biggest_addition"),
        F.col("biggest_removal")
    )

edit_size_query = edit_size_tumbling_df \
    .writeStream \
    .format("console") \
    .outputMode("update") \
    .option("truncate", False) \
    .trigger(processingTime="30 seconds") \
    .start()

## Exploration 2 - Overlapping window: most active pages

Windows of 5 minutes that slide every 1 minute: the windows **overlap**, so each event is counted in 5 windows.
We keep the pages changed at least twice in a window.

In [ ]:
hot_pages_sliding_df = events_df \
    .groupBy(
        F.window(F.col("event_time"), "5 minutes", "1 minute"),
        F.col("title")
    ) \
    .agg(F.count("*").alias("change_count")) \
    .filter(F.col("change_count") >= 2) \
    .select(
        F.lit("2_sliding_hot_pages").alias("exploration"),
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        F.col("title"),
        F.col("change_count")
    )

hot_pages_query = hot_pages_sliding_df \
    .writeStream \
    .format("console") \
    .outputMode("update") \
    .option("truncate", False) \
    .trigger(processingTime="30 seconds") \
    .start()

## Exploration 3 - Different window time: activity by type of change

Same idea as the demo, but with a 10 minute tumbling window instead of 1 hour,
and grouped by type of change (`edit`, `new`, `log`, `categorize`) instead of `bot`.

In [ ]:
type_counts_df = events_df \
    .groupBy(
        F.window(F.col("event_time"), "10 minutes"),
        F.col("type")
    ) \
    .agg(
        F.count("*").alias("change_count"),
        F.approx_count_distinct("user").alias("distinct_users")
    ) \
    .select(
        F.lit("3_tumbling_change_type").alias("exploration"),
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        F.col("type"),
        F.col("change_count"),
        F.col("distinct_users")
    )

type_counts_query = type_counts_df \
    .writeStream \
    .format("console") \
    .outputMode("update") \
    .option("truncate", False) \
    .trigger(processingTime="30 seconds") \
    .start()

## Stop the streams

Run this cell to stop all the running queries (do it before re-running a cell above).

In [ ]:
for query in spark.streams.active:
    query.stop()